# CHROMIA AI : entraînement et évaluation sur Colab

Ce notebook relance la chaîne complète du dépôt (préparation des données, entraînement, évaluation, statistiques) sur un GPU Colab.
Activez d'abord le GPU : **Exécution > Modifier le type d'exécution > GPU**.

À renseigner avant de lancer : l'adresse du dépôt (cellule 2) et l'emplacement des photos (cellule 4).

In [ ]:
import torch
print("GPU disponible :", torch.cuda.is_available())
!nvidia-smi -L

In [ ]:
# 1. Récupérer le code
REPO_URL = "https://github.com/VOTRE_COMPTE/chromia-ai.git"   # À MODIFIER
!git clone $REPO_URL chromia-ai
%cd chromia-ai
!python -m pip install -q -r requirements.txt

In [ ]:
# 2. Monter Google Drive (les photos et les poids y sont conservés entre deux sessions)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Emplacement des photos en couleurs (un seul dossier, sans sous-dossiers)
PHOTOS = "/content/drive/MyDrive/photos_brutes"   # À MODIFIER
!ls "$PHOTOS" | head -5
!ls "$PHOTOS" | wc -l

In [ ]:
# 4. Préparer les données : contrôle qualité et découpage 80/10/10 (graine 42)
!python prepare_data.py --src "$PHOTOS" --out data_split --split 80 10 10 --seed 42
!cat data_split/dataset_report.json

In [ ]:
# 5a. Entraînement sur GPU : valeurs par défaut du projet (256 px, 40 époques)
!python train.py --data-dir data_split/train --val-dir data_split/val --size 256 --epochs 40

# 5b. Pour retrouver exactement la configuration du rapport (128 px, 20 époques), utilisez plutôt :
# !python train.py --data-dir data_split/train --val-dir data_split/val --size 128 --epochs 20 --base 32 --workers 0

In [ ]:
# 6. Évaluation sur le jeu de test (référence grise, MC-Dropout 8 passes)
!python evaluate.py --data-dir data_split/test --ckpt checkpoints/best.pt --out results --mc 8
!python analyze_results.py --csv results/per_image.csv --out results --splits data_split/splits.json

In [ ]:
# 7. Afficher les résultats
from IPython.display import Image, Markdown, display
display(Markdown(open('results/summary.md').read()))
display(Image('checkpoints/history.png'))
display(Image('results/qualitative.png'))

In [ ]:
# 8. Sauvegarder les poids et les résultats dans Drive
!mkdir -p /content/drive/MyDrive/chromia_ai_sorties
!cp checkpoints/best.pt checkpoints/history.json /content/drive/MyDrive/chromia_ai_sorties/
!cp -r results /content/drive/MyDrive/chromia_ai_sorties/